In [0]:
import pyspark.sql.functions as F
import json

## Run ID Selection

In [0]:
def list_ingestion_dates(base_path: str):
    items = dbutils.fs.ls(base_path)
    dates = []
    for i in items:
        if i.name.startswith("ingestion_date="):
            dates.append(i.name.replace("ingestion_date=", "").replace("/", ""))
    return sorted(dates)

def list_run_ids(day_path: str):
    items = dbutils.fs.ls(day_path)
    runs = []
    for i in items:
        if i.name.startswith("run_id="):
            runs.append(i.name.replace("run_id=", "").replace("/", ""))
    return sorted(runs)

def latest_successful_run_for_day(day_path: str):
    """
    Retorna (run_id, run_path) do run mais recente bem-sucedido naquele dia.
    Prioridade:
      1) Se existe day/_LATEST e o run apontado tem _SUCCESS => usa ele
      2) Caso contrário, pega o maior run_id que tenha run/_SUCCESS
      3) Se nenhum run válido => retorna (None, None)
    """
    latest_ptr = f"{day_path}/_LATEST"
    if path_exists(latest_ptr):
        run_id = read_text(latest_ptr)
        run_path = f"{day_path}/run_id={run_id}"
        if path_exists(f"{run_path}/_SUCCESS"):
            return run_id, run_path

    # fallback: busca maior run_id com _SUCCESS
    run_ids = list_run_ids(day_path)
    for run_id in reversed(run_ids):  # do maior para o menor
        run_path = f"{day_path}/run_id={run_id}"
        if path_exists(f"{run_path}/_SUCCESS"):
            return run_id, run_path

    return None, None

def pick_latest_valid_day_and_run(base_path: str):
    """
    Retorna (ingestion_date, run_id, run_path) do dia+run mais recentes válidos.
    Estratégia: varre dias do mais recente para o mais antigo até achar um run válido.
    """
    ingestion_dates = list_ingestion_dates(base_path)

    for d in reversed(ingestion_dates):  # do mais recente para o mais antigo
        day_path = f"{base_path}/ingestion_date={d}"

        # se existir _SUCCESS no dia, ótimo (modo 1x/dia)
        # mas mesmo que não exista (modo multi-run), tentamos achar run/_SUCCESS
        run_id, run_path = latest_successful_run_for_day(day_path)
        if run_id is not None:
            return d, run_id, run_path

    raise Exception("No successful ingestion found in Bronze layer (no valid _SUCCESS markers).")

## Transformation Functions

In [0]:
BUSINESS_COLUMNS = [
    "name", "brewery_type", "address", "postal_code", "city", "state", "country",
    "latitude", "longitude", "phone", "website_url",
    "has_proper_encoding", "has_geolocation", "has_address", "has_website"
]

def standardize(df_raw):
    """
    String cleanup + canonical location/address + typing for lat/long + ingestion timestamp.
    """
    return (
        df_raw
        .withColumn("name", F.trim(F.col("name")))
        .withColumn("country", F.initcap(F.trim(F.col("country"))))
        .withColumn("city", F.initcap(F.trim(F.col("city"))))
        .withColumn("state", F.initcap(F.trim(F.coalesce(F.col("state_province"), F.col("state")))))
        .withColumn("address_1_canon", F.coalesce(F.col("address_1"), F.col("street")))
        .withColumn("address", F.concat_ws(", ", F.col("address_1_canon"), F.col("address_2"), F.col("address_3")))
        .withColumn("phone", F.regexp_replace(F.trim(F.col("phone")), r"[^\d+]", ""))
        .withColumn("latitude", F.round(F.col("latitude").cast("double"), 6))
        .withColumn("longitude", F.round(F.col("longitude").cast("double"), 6))
        .withColumn("ingestion_timestamp", F.current_timestamp())
    )

def add_quality_flags(df_standardized):
    """
    has_proper_encoding + has_website/has_address/has_geolocation
    """
    text_cols = ["name", "address", "country", "city", "state"]  # evite street aqui; já canonizamos

    condition = None
    for c in text_cols:
        expr = F.coalesce(F.col(c), F.lit("")).contains("\uFFFD")  # garante boolean (não NULL)
        condition = expr if condition is None else (condition | expr)

    return (
        df_standardized
        .withColumn("has_proper_encoding", ~condition)
        .withColumn("has_website", F.col("website_url").isNotNull())
        .withColumn("has_address", F.col("address").isNotNull())
        .withColumn("has_geolocation", F.col("latitude").isNotNull() & F.col("longitude").isNotNull())
    )

def add_hash(df_flagged):
    """
    Create deterministic hash over business columns to detect changes.
    """
    return df_flagged.withColumn(
        "hash_merge",
        F.sha2(
            F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in BUSINESS_COLUMNS]),
            256
        )
    )

def build_silver_df(df_hash):
    """
    Select final columns.
    """
    return df_hash.select(
        "id", "name", "brewery_type", "address", "postal_code", "city", "state", "country",
        "latitude", "longitude", "phone", "website_url",
        "has_proper_encoding", "has_geolocation", "has_address", "has_website",
        "ingestion_timestamp", "hash_merge"
    )